# Correlation and Hypothesis Testing: English vs Maths
**Main question:** Do students who do well in English also do well in Maths?

Dataset: English and Maths marks of 10 students.

## Part A: Warm-up (Theory)

**Parametric vs non-parametric**
- *Parametric* methods assume the data follow a known distribution (usually normal) and use the actual values (means, standard deviations).
- *Non-parametric* methods make no distribution assumption and usually work on ranks/order, so they are robust to outliers.

**Spearman's correlation** is **non-parametric** (uses ranks, measures a *monotonic* relationship).
**Pearson's correlation** is **parametric** (uses actual marks, needs roughly normal data, measures a *linear* relationship).

## Part B: Spearman's rank correlation

In [1]:
import pandas as pd
from scipy import stats

data = pd.DataFrame({
    "English": [56, 75, 45, 71, 62, 64, 58, 80, 76, 61],
    "Maths":   [66, 70, 40, 60, 65, 56, 59, 77, 67, 63]
})
data.index = range(1, 11)   # Student 1..10
data

,English,Maths
1,56,66
2,75,70
3,45,40
4,71,60
5,62,65
6,64,56
7,58,59
8,80,77
9,76,67
10,61,63


Rank the marks. As the worksheet says, **rank 1 = highest mark** (`ascending=False`).

In [2]:
data["Eng_Rank"]   = data["English"].rank(ascending=False)
data["Maths_Rank"] = data["Maths"].rank(ascending=False)

data["d"]  = data["Eng_Rank"] - data["Maths_Rank"]
data["d2"] = data["d"] ** 2
data

,English,Maths,Eng_Rank,Maths_Rank,d,d2
1,56,66,9.0,4.0,5.0,25.0
2,75,70,3.0,2.0,1.0,1.0
3,45,40,10.0,10.0,0.0,0.0
4,71,60,4.0,7.0,-3.0,9.0
5,62,65,6.0,5.0,1.0,1.0
6,64,56,5.0,9.0,-4.0,16.0
7,58,59,8.0,8.0,0.0,0.0
8,80,77,1.0,1.0,0.0,0.0
9,76,67,2.0,3.0,-1.0,1.0
10,61,63,7.0,6.0,1.0,1.0


In [3]:
n = len(data)
sum_d2 = data["d2"].sum()
rs = 1 - (6 * sum_d2) / (n * (n**2 - 1))

print("Sum of d2 =", sum_d2)
print("rs (formula) =", round(rs, 4))
print("rs (pandas)  =", round(data["English"].corr(data["Maths"], method="spearman"), 4))

Sum of d2 = 54.0
rs (formula) = 0.6727
rs (pandas)  = 0.6727


**Answer:** rs = 1 − 6(54)/(10 × 99) = **0.673**. The formula and pandas agree.
The relationship is **positive** (high English rank goes with high Maths rank) and **moderate to strong** (0.6 to 0.8).

*Note: ranking from highest or lowest only flips the sign of d. d² and rs stay the same.*

## Part C: Pearson's correlation

In [4]:
print("Mean English =", data["English"].mean())
print("Mean Maths   =", data["Maths"].mean())

r = data["English"].corr(data["Maths"], method="pearson")
print("Pearson r =", round(r, 4))
print("Spearman rs =", round(rs, 4))

Mean English = 64.8
Mean Maths   = 62.3
Pearson r = 0.8059
Spearman rs = 0.6727


**Which is bigger?** r (0.806) is bigger than rs (0.673).

**Why Student 3 (45, 40) makes r bigger:** this student is far below the mean in *both* subjects. Pearson uses actual distances from the mean, so this one point sits far out along the trend line and pulls r up. Spearman only sees "lowest in both" (rank 10 and 10), so it gets no extra boost.

In [5]:
test = data[["English", "Maths"]].copy()
test.loc[3] = [15, 10]          # change Student 3

r_new  = test["English"].corr(test["Maths"], method="pearson")
rs_new = test["English"].corr(test["Maths"], method="spearman")

print(f"Pearson : before {r:.4f}  -> after {r_new:.4f}")
print(f"Spearman: before {rs:.4f} -> after {rs_new:.4f}")

Pearson : before 0.8059  -> after 0.9378
Spearman: before 0.6727 -> after 0.6727


**Which changed?** **Pearson r changed** (0.806 to 0.938). **Spearman did not** (0.673).
Student 3 is still the lowest in both subjects, so the ranks are identical, but the marks are now more extreme, so Pearson reacts. This shows Pearson is **sensitive to outliers** and Spearman is **robust**.

## Part D: Hypothesis testing

- **H₀:** There is no relationship between English and Maths marks in the population (correlation = 0).
- **H₁:** There is a relationship between English and Maths marks (correlation ≠ 0), two-tailed.
- **Significance level:** α = 0.05. Critical t for df = 8 is 2.306.

### Pearson test (parametric)
First check normality with Shapiro-Wilk (H₀: data are normal).

In [6]:
for col in ["English", "Maths"]:
    stat, p = stats.shapiro(data[col])
    print(f"{col}: W = {stat:.3f}, p = {p:.3f}")

English: W = 0.963, p = 0.820
Maths: W = 0.923, p = 0.379


Both p-values are above 0.05, so we do not reject normality. **Yes, it is OK to use Pearson's test.** (With n = 10 the Shapiro test has low power, so treat this as a rough check.)

In [7]:
df_free = n - 2
t_p = r * ((n - 2) / (1 - r**2)) ** 0.5
r_check, p_p = stats.pearsonr(data["English"], data["Maths"])

print("t-statistic =", round(t_p, 3), " df =", df_free)
print("critical t  =", round(stats.t.ppf(0.975, df_free), 3))
print("p-value     =", round(p_p, 4))

t-statistic = 3.85  df = 8
critical t  = 2.306
p-value     = 0.0049


**Decision:** t = 3.850 > 2.306 and p = 0.0049 < 0.05, so **reject H₀**.

**Conclusion:** There is a statistically significant, strong positive linear correlation between English and Maths marks (r = 0.81, p = 0.005).

### Spearman test (non-parametric)

In [8]:
t_s = rs * ((n - 2) / (1 - rs**2)) ** 0.5
rs_check, p_s = stats.spearmanr(data["English"], data["Maths"])

print("t-statistic =", round(t_s, 3), " df =", df_free)
print("p-value     =", round(p_s, 4))

t-statistic = 2.572  df = 8
p-value     = 0.033


**Decision:** t = 2.572 > 2.306 and p = 0.033 < 0.05, so **reject H₀**.

**Conclusion:** There is a statistically significant positive monotonic relationship between English and Maths ranks (rs = 0.67, p = 0.033).

### Compare

In [9]:
print(f"Pearson  p-value = {p_p:.4f}")
print(f"Spearman p-value = {p_s:.4f}")

Pearson  p-value = 0.0049
Spearman p-value = 0.0330


- **Smaller p-value:** Pearson (0.005 vs 0.033).
- **Which to report:** **Pearson**, because both subjects pass the normality check and the relationship looks linear, so the parametric test is valid and more powerful. Spearman is the safer choice if normality failed or outliers were a worry. Here both tests agree.
- **What a p-value tells you:** the probability of getting a result at least this extreme *if H₀ were true*. It is *not* the probability that H₀ is true.

## Part E: Mann-Whitney U test
Split students into top 5 and bottom 5 by English mark, then compare their Maths marks.

In [10]:
s = data.sort_values("English", ascending=False)
top5, bottom5 = s.head(5), s.tail(5)

print("Top 5 English    -> Maths marks:", top5["Maths"].tolist(),    " mean =", top5["Maths"].mean())
print("Bottom 5 English -> Maths marks:", bottom5["Maths"].tolist(), " mean =", bottom5["Maths"].mean())

Top 5 English    -> Maths marks: [77, 67, 70, 60, 56]  mean = 66.0
Bottom 5 English -> Maths marks: [65, 63, 59, 66, 40]  mean = 58.6


In [11]:
t_res = stats.ttest_ind(top5["Maths"], bottom5["Maths"])
u_res = stats.mannwhitneyu(top5["Maths"], bottom5["Maths"], alternative="two-sided")

print(f"Independent t-test: t = {t_res.statistic:.3f}, p = {t_res.pvalue:.3f}")
print(f"Mann-Whitney U    : U = {u_res.statistic:.1f}, p = {u_res.pvalue:.3f}")

Independent t-test: t = 1.221, p = 0.257
Mann-Whitney U    : U = 18.0, p = 0.310


**Is either significant?** **No.** t-test p = 0.257 and Mann-Whitney p = 0.310, both above 0.05. Top-5 Maths mean is 66 vs 58.6 for the bottom 5, but that gap is within random variation.

**Why different from the correlation tests?**
1. Splitting into groups **throws away information**: English 64 and 80 are both just "top", and 62 and 45 both just "bottom". Correlation uses every student's exact position.
2. With only 5 students per group the tests have **very low power**.
3. Student 3's very low Maths mark (40) inflates the bottom group's spread, hiding the group difference.

## Final Answer to the Main Question

**Yes. Students who do well in English tend to do well in Maths.**

| Test | Statistic | p-value | Decision |
|---|---|---|---|
| Spearman (non-parametric) | rs = 0.673 | 0.033 | Reject H₀ |
| Pearson (parametric) | r = 0.806 | 0.005 | Reject H₀ |
| Independent t-test (top vs bottom 5) | t = 1.22 | 0.257 | Fail to reject |
| Mann-Whitney U (top vs bottom 5) | U = 18 | 0.310 | Fail to reject |

Both correlation tests show a significant positive relationship. The group-split tests missed it only because splitting loses information and the groups are tiny.
**Caution:** this is correlation on 10 students. It shows English and Maths marks go together, not that one causes the other.